# DNAscape quickstart

This notebook introduces the two core DNAscape workflows on bundled example data. In section A, `rmap` maps between replication features by fitting or deterministic transformation under explicit assumptions. In section B, `rsim` uses a firing-rate profile and fork-speed assumptions to simulate stochastic DNA replication.


## Setup

Use an environment where DNAscape is installed, or start Jupyter from the repository root after an editable install. Autoreload keeps the notebook responsive while the source code is being developed.


In [ ]:
%reload_ext autoreload
%autoreload 2

from dnascape import *

## Load example data

We start from the bundled hESC chromosome 1 replication-timing profile. The loaded profile is a one-dimensional array with one value per genomic bin. `select_profile_region` returns the selected timing values together with `positions_kb`, the genomic coordinate of each selected bin. The example data are sampled at 1 kb, which is the default raw spacing.


In [ ]:
chr_number = 1
interval_kb = (0, 60_001)
fork_speed = 1.4

replication_timing_full = loadcsv(
    "example",
    example="replication_timing_hESC",
    chr_number=chr_number,
)

replication_timing, positions_kb = select_profile_region(
    replication_timing_full,
    interval_kb=interval_kb,
)

print(f"Loaded: replication timing profile of hESC (chromosome {chr_number}) with {len(replication_timing)} values, covering positions {int(positions_kb[0])} kb to {int(positions_kb[-1])} kb.")

A quick plot is useful before fitting. The y-axis is inverted because earlier replication appears higher in standard replication-timing visualisations.


In [ ]:
plotf(
    replication_timing,
    x_array=positions_kb,
    invyQ=True,
    xtitle="Chromosome position (kb)",
    ytitle="Replication timing (min)",
)

## A. Mapping replication features

`rmap` is the mapping interface. It takes a source feature, a target feature, and a dictionary of assumptions. Internally, DNAscape selects the appropriate route for the requested mapping.


### Fit firing rate from timing

Here we infer a space-dependent firing-rate profile from replication timing, assuming a constant fork speed and the selected genomic resolution.


In [ ]:
firing_rate = rmap(
    source={
        "replication_timing": replication_timing,
        "fork_speed": fork_speed,
    },
    target="firing_rate",
)

print(f"Fitted I(x) from replication timing data, assuming a constant fork speed of {fork_speed} kb/min.")

In [ ]:
plotf(
    firing_rate,
    x_array=positions_kb,
    xtitle="Chromosome position (kb)",
    ytitle="Firing rate",
)

### Map the fitted profile back to timing

Forward-mapping the fitted firing-rate profile back to timing gives a direct visual check of the mapping loop, this can later be ccompared with averaged stochastic simulations.


In [ ]:
replication_timing_mapped = rmap(
    source={
        "firing_rate": firing_rate,
        "fork_speed": fork_speed,
    },
    target="replication_timing",
)

plotf(
    replication_timing,
    replication_timing_mapped,
    x_array=positions_kb,
    labels=["Timing data (Repli-seq)", "Timing mapped from fitted firing rate"],
    invyQ=True,
    xtitle="Chromosome position (kb)",
    ytitle="Replication timing (min)",
)

As expected, the fitting is not perfect, as the constant fork speed assumptions is too strong to describe the data. We may, for example, check a measure of the misfit by looking at their difference.

In [ ]:
plotf(
    replication_timing - timing_from_firing,
    x_array=positions_kb,
    xtitle="Chromosome position (kb)",
    ytitle="Timing residual (min)",
)

### Map timing to fork directionality

Fork directionality is another view of the same replication programme, and can be obtained directly from replication timing, under a constant fork speed assumption. We use the smoother timing profile generated from the fitted firing rate.


In [ ]:
timing_reference = timing_from_firing

fork_directionality = rmap(
    source={
        "replication_timing": timing_reference,
        "fork_speed": fork_speed,
    },
    target="fork_directionality",
)

plotf(
    fork_directionality,
    x_array=positions_kb,
    xtitle="Chromosome position (kb)",
    ytitle="Fork directionality",
    ylims=(-1.05, 1.05),
)

## B. Simulating replication

`rsim` is the generative interface. It simulates replication from a supplied firing-rate profile, fork speed, and optional assumptions such as stalling, firing factors, or periodic geometry. See `3_simulation.ipynb` for the full scope of its properties.


### Simulate from the fitted firing rate

The simulation uses the fitted firing-rate profile from section A. `rsim` prints progress by default.


In [ ]:
sim_number = 1

simulation = rsim(
    source={
        "firing_rate": firing_rate,
        "fork_speed": fork_speed,
    },
    assumptions={
        "sim_number": sim_number,
        "time_statsQ": True,
    },
)

In [ ]:
plotf(
    replication_timing,
    replication_timing_mapped,
    simulation["replication_timing"],
    x_array=positions_kb,
    labels=["Data (Repli-seq)", "Map (from firing rate fitting)", f"Simulation (mean over {sim_number} runs)"],
    invyQ=True,
    xtitle="Chromosome position (kb)",
    ytitle="Replication timing (min)",
)

### Inspect simulation outputs

The simulation returns genome-wide profiles, time courses, and per-simulation summaries. These plots show the replicated fraction over time, origin efficiency along the genome, inter-origin distances, and compact summary statistics.


In [ ]:
plotf(
    simulation["replicated_fraction"],
    simulation["active_forks"],
    labels=["Replicated fraction", "Active forks"],
    dual_axis=True,
    xtitle="Time (min)",
)

In [ ]:
plotf(
    simulation["origin_efficiency"],
    firing_rate,
    x_array=positions_kb,
    labels=["Origin efficiency", "Firing rate"],
    dual_axis=True,
    xtitle="Chromosome position (kb)",
)

In [ ]:
ploth(
    simulation["inter_origin_distance"],
    xtitle="Inter-origin distance (kb)",
    bin_size=output_resolution_kb,
    xmax=1000,
)

In [ ]:
summary = {
    "mean_completion_time_min": sum(simulation["completion_time"]) / len(simulation["completion_time"]),
    "mean_num_origins": sum(simulation["num_origins"]) / len(simulation["num_origins"]),
    "mean_origin_efficiency": sum(simulation["origin_efficiency"]) / len(simulation["origin_efficiency"]),
}

summary

### Combined genomic view

`mplotf` stacks several genomic tracks into one figure. This is often the most compact way to compare the input data, fitted maps, and simulation-derived profiles over the same interval.


In [ ]:
mplotf(
    [
        [replication_timing, timing_from_firing, simulation["replication_timing"]],
        firing_rate,
        fork_directionality,
        [simulation["origin_efficiency"], firing_rate],
    ],
    mlabels=[
        ["Timing data", "Deterministic map", "Mean simulation"],
        [None],
        [None],
        ["Origin efficiency", "Firing rate"],
    ],
    mx_array=[positions_kb, positions_kb, positions_kb, positions_kb],
    mdual_axis=[False, False, False, True],
    minvyQ=[True, False, False, False],
    mylims=[(None, None), (None, None), (-1.05, 1.05), (None, None)],
    mxtitle=["", "", "", "Chromosome position (kb)"],
    mytitle=[
        "Replication timing (min)",
        "Firing rate",
        "Fork directionality",
        "Simulation output",
    ],
    figsize=(10, 2.2),
    hspace=0.18,
)